In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from ctgan import CTGAN
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# 1. 数据加载和预处理
df = pd.read_excel('../data/new_data_x_names.xlsx')
print(f"原始数据集大小: {df.shape}")

# 分离特征和多个目标变量
X = df.drop(['x25', 'x26', 'x27', 'x28', 'x29'], axis=1)
y = df[['x25', 'x26', 'x27', 'x28', 'x29']]

# 2. 使用IQR方法检测所有异常值并计算其异常程度
def find_outliers_with_severity(df, target_columns, threshold=1.5):
    """
    使用IQR方法检测所有异常值，计算异常程度并返回排序后的异常值索引
    threshold: IQR的倍数，通常为1.5
    """
    df_clean = df.copy()
    outlier_data = []
    
    for col in target_columns:
        # 计算Q1, Q3和IQR
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # 计算异常值边界
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        # 找出超出下界限的行
        lower_outliers = df_clean[df_clean[col] < lower_bound]
        for idx in lower_outliers.index:
            # 计算异常程度：与边界的差距除以IQR
            severity = abs(df_clean.loc[idx, col] - lower_bound) / IQR
            outlier_data.append((idx, col, severity))
        
        # 找出超出上界限的行
        upper_outliers = df_clean[df_clean[col] > upper_bound]
        for idx in upper_outliers.index:
            # 计算异常程度：与边界的差距除以IQR
            severity = abs(df_clean.loc[idx, col] - upper_bound) / IQR
            outlier_data.append((idx, col, severity))
    
    # 按严重程度排序并去重索引
    outlier_data.sort(key=lambda x: x[2], reverse=True)
    unique_indices = []
    seen = set()
    for idx, col, severity in outlier_data:
        if idx not in seen:
            unique_indices.append((idx, severity))
            seen.add(idx)
    
    print(f"检测到 {len(unique_indices)} 行含有异常值")
    
    return unique_indices

# 找出所有异常值行
target_cols = ['x25', 'x26', 'x27', 'x28', 'x29']
all_outliers_with_severity = find_outliers_with_severity(df, target_cols)

# 移除所有异常值行（不限制为27个）
outliers_to_remove = [idx for idx, _ in all_outliers_with_severity]
print(f"将移除所有 {len(outliers_to_remove)} 个异常值")

# 移除选定的异常值行
df_clean = df.drop(outliers_to_remove)
print(f"清除异常值后的数据集大小: {df_clean.shape}")

# 重新分离特征和目标变量
X_clean = df_clean.drop(['x25', 'x26', 'x27', 'x28', 'x29'], axis=1)
y_clean = df_clean[['x25', 'x26', 'x27', 'x28', 'x29']]

# 3. 可视化异常值检测前后的分布对比 - 箱线图
fig, axes = plt.subplots(len(target_cols), 2, figsize=(15, 4*len(target_cols)))
for i, col in enumerate(target_cols):
    # 异常值检测前
    sns.boxplot(x=df[col], ax=axes[i, 0])
    axes[i, 0].set_title(f'{col} - 去除异常值前')
    
    # 异常值检测后
    sns.boxplot(x=df_clean[col], ax=axes[i, 1])
    axes[i, 1].set_title(f'{col} - 去除异常值后')

plt.tight_layout()
plt.show()

# 4. 绘制清理后的目标变量折线图
plt.figure(figsize=(15, 10))
for i, col in enumerate(target_cols):
    plt.subplot(len(target_cols), 1, i+1)
    plt.plot(y_clean.index, y_clean[col], marker='o', linestyle='-', markersize=2)
    plt.title(f'{col} - 清理后的数据分布')
    plt.ylabel(col)
    plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# 5. 保存清理后的数据集
output_path = '../data/quyi_data_x_names.xlsx'
df_clean.to_excel(output_path, index=False)
print(f"已移除 {len(outliers_to_remove)} 个异常值行")
print(f"清理后的数据集已保存至: {output_path}")

# 6. 打印清理前后的数据统计信息
print("\n清理前的目标变量统计信息:")
print(df[target_cols].describe())

print("\n清理后的目标变量统计信息:")
print(df_clean[target_cols].describe())